
# Árboles y ensambles, Notebook 6 (bonus track)
## C5.0 con `c50py`: gain ratio, categorías sin dummies, valores faltantes, poda pesimista y boosting incorporado

**Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De qué se trata

Al final del Notebook 2 quedó dicho que después de ID3 la historia se bifurcó en C4.5 y CART. Todo lo que vino después siguió la rama
de CART: Gini, cortes binarios sobre números, bagging, random forest, gradient boosting. Este
bonus track recorre la otra rama, la de Quinlan mismo: **C4.5** (1993) y su versión comercial
**C5.0** (también llamada See5), que durante años fue el árbol de referencia en la industria y que
en R se usa con el paquete `C50`. Quinlan liberó el código fuente en C de C5.0 en 2011, y a
partir de ese código David Díaz, autor de este material, escribió **`c50py`** (`pip install c50py`),
la primera implementación de C5.0 en Python: desde cero en Python y NumPy, con la interfaz de
scikit-learn (entra en un `Pipeline` o un `GridSearchCV`), y con cosas que ni el C5.0 original ni
el paquete `C50` de R, que solo envuelve ese código, ofrecen: un `C5Regressor`, la regla exacta
que se aplicó a cada predicción y exportación a Graphviz. Este notebook usa la versión 0.3.0, que
trae la poda pesimista y la selección de cortes tal como están en el código de C4.5.

**Lo que importa para el negocio** son cuatro cosas que C5.0 hace y que con scikit-learn hay que
resolver a mano: trabaja con variables numéricas y categóricas a la vez sin preprocesamiento;
cuando una categórica tiene muchos valores, agrupa los valores en el corte en vez de necesitar una
columna de ceros y unos por valor; maneja los valores faltantes sin imputar; y para cada cliente
devuelve la regla exacta que se activó. Con `trials = 1` el resultado es un árbol más chico y más
legible que el de scikit-learn con desempeño igual o mejor; con `trials = 10` se convierte en un
ensamble boosteado sin cambiar nada más. Empezamos por ahí, con un caso de negocio, y después
abrimos cada mecanismo.

| | CART (scikit-learn) | C5.0 (`c50py`) |
|---|---|---|
| Criterio | Gini o varianza | **gain ratio** (ganancia / entropía de la partición) |
| Categorías | hay que convertirlas en dummies | se usan directo: cortes por **subconjuntos** de valores |
| Valores faltantes | hay que imputar antes | la observación baja por **las dos ramas** con pesos |
| Poda | costo-complejidad con validación | **pesimista**, solo con entrenamiento, con un factor de confianza `cf` |
| Boosting | librería aparte | incorporado: `trials = 10` |
| Salida | el árbol | el árbol, las **reglas** en texto, y la regla que se aplicó a cada predicción |

### Qué vas a aprender hoy

1. Un caso de fuga de clientes con variables numéricas y categóricas, huecos, y la regla que se activa por cliente: `c50py` contra scikit-learn, árbol contra árbol y ensamble contra ensamble.
2. Por qué C5.0 parte las categorías en dos subconjuntos, y qué es el gain ratio.
3. Qué hace con un valor faltante, al entrenar y al predecir.
4. Cómo poda sin datos de validación, con la cuenta de C4.5 a mano.
5. El boosting incorporado y las reglas exportadas.
6. Qué da todo esto sobre nuestras 219 empresas, comparado con CART.


In [ ]:

import numpy as np
import pandas as pd
import warnings
import itertools

try:
    import c50py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "c50py>=0.3.0"], check=True)
    import c50py
from c50py import C5Classifier, C5Regressor
assert tuple(int(v) for v in c50py.__version__.split(".")[:2]) >= (0, 3), "Este notebook necesita c50py 0.3.0 o superior: pip install --upgrade c50py"
print("c50py", c50py.__version__)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("c50py listo.")



## 1. El caso de negocio: fuga de clientes

Una empresa de telecomunicaciones con 3.000 clientes quiere saber quién se va a ir, y por qué,
para hacer algo antes. De cada cliente sabe la **región** (16 regiones de Chile), el **plan**
(prepago, postpago, empresa), la antigüedad en meses, el gasto mensual, los reclamos en los
últimos 12 meses, los gigabytes de datos que usa y la edad. Los datos son simulados con una
semilla fija (el código está abajo, así que el ejemplo es reproducible y se puede modificar): la
fuga depende sobre todo de un grupo de regiones con mala cobertura, de los reclamos, de la
antigüedad y del plan, y el 8% de los clientes no tiene registrado el uso de datos ni la edad.


In [ ]:

def simular_churn(n=3000, semilla=2026):
    rng = np.random.default_rng(semilla)
    REGIONES = ["Arica y Parinacota", "Tarapacá", "Antofagasta", "Atacama", "Coquimbo", "Valparaíso", "Metropolitana", "O'Higgins",
                "Maule", "Ñuble", "Biobío", "La Araucanía", "Los Ríos", "Los Lagos", "Aysén", "Magallanes"]
    pesos = np.array([1, 1, 2, 1, 2, 5, 12, 2, 2, 1, 4, 2, 1, 2, 0.5, 0.5]); pesos = pesos / pesos.sum()
    region = rng.choice(REGIONES, n, p=pesos)
    plan = rng.choice(["prepago", "postpago", "empresa"], n, p=[0.45, 0.45, 0.10])
    antig = rng.integers(1, 120, n)
    gasto = np.round(np.where(plan == "empresa", rng.normal(45000, 12000, n), np.where(plan == "postpago", rng.normal(25000, 8000, n), rng.normal(9000, 4000, n))).clip(2000), -2)
    reclamos = rng.poisson(0.7, n)
    datos = np.round(rng.gamma(2, 4, n), 1)
    edad = rng.integers(18, 80, n)
    cobertura_mala = np.isin(region, ["Aysén", "Magallanes", "Arica y Parinacota", "Tarapacá", "Los Ríos", "La Araucanía"])
    logit = -2.0 + 2.2 * cobertura_mala + 0.9 * reclamos - 0.02 * antig + 1.1 * (plan == "prepago") - 0.6 * (plan == "empresa") - 0.03 * datos + 0.8 * (antig < 6)
    churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame(dict(region=region, plan=plan, antiguedad_meses=antig, gasto_mensual=gasto, reclamos_12m=reclamos, datos_gb=datos, edad=edad, churn=churn))
    for c in ["datos_gb", "edad"]:                      # 8% de huecos
        df.loc[rng.random(n) < 0.08, c] = np.nan
    return df

churn = simular_churn()
FEATS = [c for c in churn.columns if c != "churn"]
tr, te = churn.iloc[:2000], churn.iloc[2000:]
print(f"Clientes: {len(churn)}   fuga: {churn.churn.mean():.1%}   huecos: datos_gb {churn.datos_gb.isna().sum()}, edad {churn.edad.isna().sum()}")
print("\nFuga por región:")
print(churn.groupby("region").churn.mean().round(2).sort_values().to_string())
churn.head(8)



**Dos preparaciones para los mismos datos.** Para `c50py`, la tabla tal cual: siete columnas, dos de
ellas texto, con sus huecos. Para scikit-learn, lo mínimo que exige: las dos columnas de texto
convertidas en columnas de ceros y unos (una por región y una por plan, 24 en total) y los huecos
rellenados con la mediana del entrenamiento. Comparamos con el mismo freno en los dos (mínimo de
clientes por hoja), y además buscamos los mejores hiperparámetros de scikit-learn por validación
cruzada, para que nadie diga que el CART quedó mal ajustado.

Usamos los valores por defecto de `c50py` (poda pesimista con `cf = 0,25`) y solo variamos el
mínimo de clientes por hoja, el mismo freno en los dos. Agregamos además una fila de `c50py` sin
poda, por una razón que se explica en la sección 5: la poda de C4.5 mira el error, no el orden,
y para un ranking de riesgo (AUC) conviene desactivarla.


In [ ]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
import time

X_tr, X_te = tr[FEATS].values.astype(object), te[FEATS].values.astype(object)
Xd = pd.get_dummies(churn[FEATS], columns=["region", "plan"]).astype(float)
mediana = Xd.iloc[:2000].median(); Xd = Xd.fillna(mediana)
Xd_tr, Xd_te = Xd.iloc[:2000], Xd.iloc[2000:]
print("Columnas para scikit-learn después de dummies:", Xd.shape[1], "   para c50py:", len(FEATS))

def profundidad(nodo): return 0 if nodo.is_leaf else 1 + max(profundidad(nodo.children["left"]), profundidad(nodo.children["right"]))
filas = []
for msl in [10, 25]:
    t0 = time.time(); m = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=msl).fit(X_tr, tr.churn.values, feature_names=FEATS); t = time.time() - t0
    filas.append((f"c50py, poda por defecto, min_samples_leaf={msl}", "ninguno", len(m.export_rules()), profundidad(m.tree_), m.score(X_te, te.churn.values), roc_auc_score(te.churn, m.predict_proba(X_te)[:, 1]), round(t, 1)))
    if msl == 25: c5_churn = m
    t0 = time.time(); s = DecisionTreeClassifier(min_samples_leaf=msl, random_state=0).fit(Xd_tr, tr.churn); t = time.time() - t0
    filas.append((f"scikit-learn, min_samples_leaf={msl}", "dummies + mediana", s.get_n_leaves(), s.get_depth(), s.score(Xd_te, te.churn), roc_auc_score(te.churn, s.predict_proba(Xd_te)[:, 1]), round(t, 1)))
    if msl == 25: sk_churn = s
t0 = time.time(); m = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=50, pruning=False).fit(X_tr, tr.churn.values, feature_names=FEATS); t = time.time() - t0
filas.append(("c50py, sin poda, min_samples_leaf=50 (para ordenar por riesgo)", "ninguno", len(m.export_rules()), profundidad(m.tree_), m.score(X_te, te.churn.values), roc_auc_score(te.churn, m.predict_proba(X_te)[:, 1]), round(t, 1)))
gs = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [3, 4, 5, 6, 8, None], "min_samples_leaf": [10, 25, 50, 100]}, cv=5, scoring="roc_auc").fit(Xd_tr, tr.churn)
s = gs.best_estimator_
filas.append((f"scikit-learn, mejor por validación cruzada {gs.best_params_}", "dummies + mediana", s.get_n_leaves(), s.get_depth(), s.score(Xd_te, te.churn), roc_auc_score(te.churn, s.predict_proba(Xd_te)[:, 1]), 0))
tabla = pd.DataFrame(filas, columns=["modelo", "preprocesamiento", "hojas", "profundidad", "acierto en prueba", "AUC en prueba", "segundos"])
tabla["acierto en prueba"] = tabla["acierto en prueba"].round(3); tabla["AUC en prueba"] = tabla["AUC en prueba"].round(3)
print(tabla.to_string(index=False))
print(f"\nPredecir 'se queda' para todos: {1 - te.churn.mean():.3f}")



**Cómo leer la tabla.** Con el mismo freno, `c50py` da un árbol mucho más chico y mejor: con mínimo
25 por hoja, 12 hojas contra 55, más acierto y más AUC; con mínimo 10, 19 hojas contra 115. El
mejor scikit-learn por validación cruzada llega a 33 hojas y empata en AUC con el C5.0 de 12,
acertando menos. El AUC mide si el modelo ordena bien a los clientes por riesgo; el acierto,
cuántos clasifica bien con el umbral de 0,5. La fila sin poda muestra el otro camino: 27 hojas y
el mejor AUC de los árboles solos, porque conserva hojas que no cambian la clase pero afinan la
probabilidad. Y la columna de preprocesamiento es la que un equipo de negocio nota primero: nada,
contra dummies más imputación.

**Por qué scikit-learn casi no usa la región.** Con 16 columnas de ceros y unos, cada una pregunta
"¿es de la región X?" y separa un grupo chico del resto: la reducción de impureza de cualquiera de
ellas es pequeña y el árbol prefiere cortes en antigüedad o gasto. C5.0 puede preguntar por un
subconjunto de regiones en un solo corte, junta las que se comportan parecido, y la región pasa a
ser la primera pregunta. La señal estaba en los datos; lo que cambió es si el algoritmo podía
verla en una pregunta. Compara los dos árboles con mínimo 25 por hoja:


In [ ]:

print("c50py (12 hojas):\n")
c5_churn.print_tree(feature_names=FEATS)


In [ ]:

from sklearn.tree import export_text
print("scikit-learn (55 hojas), sobre las 24 columnas:\n")
print(export_text(sk_churn, feature_names=list(Xd.columns)))
usa_region = sum(1 for f in sk_churn.tree_.feature if f >= 0 and Xd.columns[f].startswith("region_"))
print("Cortes que usan alguna región en el árbol de scikit-learn:", usa_region)



**Los dos árboles dibujados.** El de `c50py` sale de `export_graphviz` (necesita el programa
Graphviz, que Colab ya trae); el de scikit-learn, de `plot_tree`. Mira el tamaño y qué pregunta
cada uno: región y reclamos arriba en el de C5.0; antigüedad, gasto y edad en el de scikit-learn,
con las regiones de a una y abajo.


In [ ]:

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
try:
    import graphviz
    fuente = c5_churn.export_graphviz(feature_names=FEATS, class_names=["se queda", "se va"], format="dot")   # sin archivo: devuelve el texto DOT
    display(graphviz.Source(fuente))
except Exception as e:
    print("Graphviz no disponible aquí:", e)
    c5_churn.print_tree()
fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(sk_churn, feature_names=list(Xd.columns), class_names=["se queda", "se va"], filled=True, fontsize=6, ax=ax, impurity=False)
ax.set_title("scikit-learn, mínimo 25 por hoja, sobre las 24 columnas: 55 hojas")
plt.show()



**La regla que se activó, cliente por cliente.** Esto es lo que convierte el árbol en una
herramienta de retención: `predict_rule` devuelve, para cada cliente, la cadena de condiciones que
lo llevó a su hoja. Cada hoja es un segmento, y cada segmento sugiere una acción distinta: a un
cliente nuevo de una región con buena cobertura no se le ofrece lo mismo que a uno de Tarapacá con
dos reclamos. Fíjate en que un dato faltante puede aparecer como condición (`datos_gb MISSING`):
C5.0 no lo imputó, lo usó como lo que es.


In [ ]:

rng = np.random.default_rng(7)
idx = rng.choice(len(te), 10, replace=False)
reglas = c5_churn.predict_rule(X_te[idx], feature_names=FEATS)
p_fuga = c5_churn.predict_proba(X_te[idx])[:, 1]
for k, i in enumerate(idx):
    c = te.iloc[i]
    print(f"Cliente {2000 + i + 1}: {c.region}, {c.plan}, {int(c.antiguedad_meses)} meses, {int(c.reclamos_12m)} reclamos, datos {c.datos_gb}, se fue: {'sí' if c.churn else 'no'}")
    print(f"   P(se va) = {p_fuga[k]:.3f}   regla: {reglas[k]}\n")


In [ ]:

# Cuántos clientes de prueba caen en cada regla, y qué fracción se fue: los segmentos de retención
segmentos = pd.DataFrame({"regla": c5_churn.predict_rule(X_te, feature_names=FEATS), "se fue": te.churn.values})
resumen = segmentos.groupby("regla")["se fue"].agg(["count", "mean"]).rename(columns={"count": "clientes de prueba", "mean": "fracción que se fue"}).sort_values("fracción que se fue", ascending=False).round(2)
pd.set_option("display.max_colwidth", 200)
print(resumen.to_string())



**Y si lo que importa es solo acertar.** `trials = 10` entrena diez árboles en serie con el esquema
de AdaBoost del Notebook 4, sin cambiar nada más, y lo comparamos con el gradient boosting de
scikit-learn sobre las columnas preprocesadas.


In [ ]:

t0 = time.time(); c5_boost = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=25, trials=10, random_state=0).fit(X_tr, tr.churn.values, feature_names=FEATS); t_c5 = time.time() - t0
t0 = time.time(); gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(Xd_tr, tr.churn); t_gb = time.time() - t0
print(pd.DataFrame([
    ("c50py, trials = 10, min_samples_leaf = 25", "ninguno", c5_boost.score(X_te, te.churn.values), roc_auc_score(te.churn, c5_boost.predict_proba(X_te)[:, 1]), round(t_c5, 1)),
    ("scikit-learn GradientBoostingClassifier (200, prof. 3, tasa 0,05)", "dummies + mediana", gb.score(Xd_te, te.churn), roc_auc_score(te.churn, gb.predict_proba(Xd_te)[:, 1]), round(t_gb, 1)),
], columns=["modelo", "preprocesamiento", "acierto en prueba", "AUC en prueba", "segundos"]).round(3).to_string(index=False))



A la par en AUC, a cambio de perder la regla por cliente. Es la decisión de siempre entre explicar
y acertar, y C5.0 la deja en un parámetro. El costo que sí se nota es el tiempo: `c50py` está
escrito en Python puro y es más lento que el C de scikit-learn; en tablas de decenas de miles de
filas conviene `numeric_threshold_strategy="quantile"` (32 cuantiles en vez de todos los umbrales) y
paciencia.

Las secciones que siguen abren el mecanismo de cada una de las cuatro cosas, con las tablas chicas
de siempre para poder contar con los dedos.



## 2. Categorías sin dummies, y el gain ratio

Volvamos a las 14 empresas del Notebook 2 para ver el mecanismo con números que se pueden contar. ID3 abría una rama por cada valor del atributo: deuda
baja, media y alta eran tres ramas. C5.0 hace algo distinto: prueba todas las formas de separar
los valores en **dos** grupos ({baja} contra {media, alta}, {media} contra {baja, alta}, y así) y
elige la mejor. Con $k$ valores hay $2^{k-1} - 1$ particiones binarias; `c50py` las prueba todas
hasta `max_categories_exhaustive` (12 por defecto) y sobre eso usa una heurística.

Para comparar cortes que parten en distinto número de pedazos, C4.5 divide la ganancia por la
entropía de la partición (*split information*):

$$
\text{gain ratio}(S) = \frac{H(Y) - \sum_g \frac{n_g}{n} H(Y \text{ en } g)}{-\sum_g \frac{n_g}{n} \log_2 \frac{n_g}{n}}
$$

El numerador es la ganancia de información del Notebook 1; el denominador es la entropía de los
tamaños de los grupos $g$ que forma el corte $S$ ($n_g$ observaciones en el grupo, $n$ en total):
vale 1 bit si el corte parte en dos mitades, 1,58 si parte en tres tercios, y crece con el número
de pedazos. Dividir castiga los cortes que ganan "gratis" por partir mucho (la trampa del
identificador).


In [ ]:

# Las 14 empresas que pidieron crédito (la tabla chica del curso)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def H(serie):
    p = pd.Series(serie).value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia_y_ratio(df, mascara_por_grupo):
    # mascara_por_grupo: lista de máscaras booleanas, una por grupo del corte
    Hy = H(df["paga"]); n = len(df)
    gan = Hy - sum(m.sum() / n * H(df.loc[m, "paga"]) for m in mascara_por_grupo)
    split_info = -sum(m.sum() / n * np.log2(m.sum() / n) for m in mascara_por_grupo if m.sum() > 0)
    return gan, split_info, gan / split_info if split_info > 0 else 0.0

filas = []
for a in ["tamano", "deuda", "garantia", "historial"]:
    vals = sorted(credito[a].unique())
    g, s, r = ganancia_y_ratio(credito, [credito[a] == v for v in vals])
    filas.append((a, "todos los valores (ID3): " + " / ".join(vals), round(g, 3), round(s, 3), round(r, 3)))
    vistos = set()
    for k in range(1, len(vals) // 2 + 1):
        for S in itertools.combinations(vals, k):
            resto = tuple(v for v in vals if v not in S)
            if frozenset(resto) in vistos: continue
            vistos.add(frozenset(S))
            m = credito[a].isin(S)
            g, s, r = ganancia_y_ratio(credito, [m, ~m])
            filas.append((a, "{" + ", ".join(S) + "} contra {" + ", ".join(resto) + "}", round(g, 3), round(s, 3), round(r, 3)))
print(pd.DataFrame(filas, columns=["atributo", "corte", "ganancia", "entropía de la partición", "gain ratio"]).to_string(index=False))



Mira la fila de deuda: partir {baja} contra {media, alta} tiene exactamente la misma ganancia que
partir en tres (0,292 bits), porque toda la información de la deuda estaba en separar a las cuatro
empresas de deuda baja, que pagan todas. Pero la partición binaria reparte 4 contra 10, con
entropía 0,863, y la de tres valores tiene entropía 1,577: el gain ratio del corte binario casi
duplica al de ID3 (0,338 contra 0,185). C5.0 obtiene lo mismo con menos pedazos, y le quedan 10
empresas juntas para seguir preguntando.

Ahora el árbol completo con `c50py`. Le decimos qué columnas son categóricas (o le pedimos que las
infiera con `infer_categorical=True`) y le pasamos los nombres para que el árbol se lea.


In [ ]:

ATRIBUTOS = ["tamano", "deuda", "garantia", "historial"]
Xc = credito[ATRIBUTOS].values.astype(object); yc = credito["paga"].values

c5_14 = C5Classifier(categorical_features=ATRIBUTOS, pruning=False, min_samples_leaf=1)
c5_14.fit(Xc, yc, feature_names=ATRIBUTOS)
c5_14.print_tree(feature_names=ATRIBUTOS)
print()
print("Reglas, una por hoja:")
for regla in c5_14.export_rules(feature_names=ATRIBUTOS):
    print("  ", regla)



Ocho hojas para 14 empresas, contra las cinco de ID3: la raíz es la misma pregunta escrita en
binario, y el resto se reparte distinto porque después de "deuda en {baja}" quedan 10 empresas
mezcladas en vez de dos grupos de 5. Como ID3, C5.0 sin freno parte hasta que cada grupo queda
puro, y con 14 empresas eso es memorizar.

**La comparación honesta con CART.** scikit-learn necesita dummies (una columna de ceros y unos por
categoría). Con validación dejando una empresa afuera cada vez (*leave-one-out*), que es lo único
que se puede hacer con 14 filas:


In [ ]:

from sklearn.model_selection import LeaveOneOut
Xd = pd.get_dummies(credito[ATRIBUTOS]).values.astype(float)
ac_c5, ac_cart = [], []
for tr, te in LeaveOneOut().split(Xd):
    m = C5Classifier(categorical_features=ATRIBUTOS, min_samples_leaf=1).fit(Xc[tr], yc[tr], feature_names=ATRIBUTOS); ac_c5.append(m.predict(Xc[te])[0] == yc[te][0])
    s = DecisionTreeClassifier(random_state=0).fit(Xd[tr], yc[tr]); ac_cart.append(s.predict(Xd[te])[0] == yc[te][0])
print(f"Acierto leave-one-out: c50py {np.mean(ac_c5):.3f}   CART con dummies {np.mean(ac_cart):.3f}")
print("Hojas del CART con dummies sobre las 14:", DecisionTreeClassifier(random_state=0).fit(Xd, yc).get_n_leaves())



C5.0 con su poda por defecto acierta 10 de 14 (71%); el CART con dummies, 8 de 14 (57%), lo mismo
que predecir "paga" para todas. Con 14 empresas esto es una anécdota, no una evidencia: dos
empresas de diferencia. Lo que sí se puede afirmar es la lectura: el árbol de C5.0 habla de
"deuda en {baja}" y no de "deuda_baja == 1", y la poda de C4.5 lo deja más chico que los ocho
hojas sin freno de arriba.



## 3. Las 219 empresas: el árbol de `c50py` contra el CART del Notebook 2

Con ratios numéricos, C5.0 busca umbrales igual que CART, con gain ratio en vez de Gini (y con
dos reglas más de C4.5: solo compiten por gain ratio los atributos con ganancia al menos promedio,
y las variables continuas pagan una pequeña penalización por la cantidad de umbrales que ofrecen).
Usamos los valores por defecto.


In [ ]:

# Las 219 empresas con ratios financieros (la tabla de impago del curso, con 5 de sus ratios).
# Está pegada aquí mismo para que el notebook no dependa de ningún archivo externo.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Empresas:", len(impago), "  Fracción en impago:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# La misma partición en todos los notebooks del set: 146 empresas para entrenar, 73 para probar
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Entrenamiento:", len(X_train), "  Prueba:", len(X_test))
print("Fracción en impago: entrenamiento", round(y_train.mean(), 3), " prueba", round(y_test.mean(), 3))


In [ ]:

Xtr, Xte = X_train.values.astype(float), X_test.values.astype(float)

def c5(**kw):
    m = C5Classifier(**kw)
    m.fit(Xtr, y_train.values, feature_names=RATIOS)
    return m

filas = []
for msl in [1, 5, 10, 20]:
    m = c5(min_samples_leaf=msl)
    filas.append((msl, len(m.export_rules(feature_names=RATIOS)), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["min_samples_leaf", "hojas", "acierto entrenamiento", "acierto prueba"]).to_string(index=False))
print()
cart2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print(f"Referencia, CART de profundidad 2 (Notebook 2): entrenamiento {cart2.score(X_train, y_train):.3f}   prueba {cart2.score(X_test, y_test):.3f}")


In [ ]:

c5_def = c5()          # valores por defecto: mínimo 1 por hoja, poda pesimista con cf = 0,25
c5_def.print_tree()



Seis hojas con los valores por defecto, 82,9% en entrenamiento y 76,7% en prueba. La raíz y el
primer nivel son los mismos del CART del Notebook 2 (ventas sobre deuda 1,278; después razón
corriente 1,866 y deuda sobre activos 0,586): gain ratio y Gini coinciden en las dos primeras
preguntas y difieren en lo que hacen con los grupos que quedan. En estas 73 empresas de prueba el
CART de dos niveles acierta 82,2% y este árbol 76,7%: cuatro empresas de diferencia, que con tan
pocos datos no dicen cuál criterio es mejor.

Lo que sí es distinto es la salida. `export_rules()` escribe cada hoja como una regla, y
`predict_rule()` devuelve, para una empresa concreta, la regla que se le aplicó: exactamente lo
que un comité de crédito pide cuando pregunta "por qué".


In [ ]:

for regla in c5_def.export_rules():
    print(regla)
print()
i = 0
print("Empresa de prueba", X_test.index[i], "  ratios:", dict(X_test.iloc[i].round(3)))
print("Regla aplicada:", c5_def.predict_rule(Xte[[i]])[0])
print("Predicción:", c5_def.predict(Xte[[i]])[0], "  real:", y_test.iloc[i])



## 4. Valores faltantes: la empresa baja por las dos ramas

En la práctica a una empresa le puede faltar un ratio. CART necesita que alguien rellene el hueco
antes, típicamente con la mediana. C5.0 no: al entrenar, si a una empresa le falta la variable por
la que pregunta un nodo, la manda a **las dos ramas**, con un peso igual a la fracción de las
empresas conocidas que fue por cada lado. Por eso en los árboles entrenados con datos incompletos
las hojas muestran conteos con decimales: empresas enteras más pedazos de empresas. Al predecir
pasa lo mismo:

$$
P(\text{impago} \mid x) = \frac{n_{\text{izq}}}{n_{\text{izq}} + n_{\text{der}}} P_{\text{izq}} + \frac{n_{\text{der}}}{n_{\text{izq}} + n_{\text{der}}} P_{\text{der}}
$$

donde $n_{\text{izq}}$ y $n_{\text{der}}$ son cuántas empresas de entrenamiento con la variable
conocida fueron por cada lado, y $P_{\text{izq}}$, $P_{\text{der}}$ lo que dice cada rama al seguir
bajando con los ratios que sí se conocen. Al elegir el corte también hay una corrección: la
ganancia se calcula con las empresas que tienen el valor y se multiplica por la fracción conocida.

Borramos al azar el 20% de los valores, en entrenamiento y en prueba, y comparamos.


In [ ]:

rng = np.random.default_rng(4)
Xm = Xtr.copy(); Xm[rng.random(Xm.shape) < 0.20] = np.nan
Xtm = Xte.copy(); Xtm[rng.random(Xtm.shape) < 0.20] = np.nan
print("Valores faltantes en entrenamiento:", int(np.isnan(Xm).sum()), "de", Xm.size, "  filas completas:", int((~np.isnan(Xm).any(axis=1)).sum()), "de 146")

c5_falt = C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values, feature_names=RATIOS)
c5_falt.print_tree(feature_names=RATIOS)


In [ ]:

mediana = np.nanmedian(Xm, axis=0)
Xi, Xti = np.where(np.isnan(Xm), mediana, Xm), np.where(np.isnan(Xtm), mediana, Xtm)
completas = ~np.isnan(Xm).any(axis=1)

comparacion = pd.DataFrame([
    ("c50py, propagación fraccionaria", C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values).score(Xtm, y_test.values)),
    ("CART, huecos rellenados con la mediana", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xi, y_train).score(Xti, y_test)),
    ("CART, solo las filas completas", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xm[completas], y_train.values[completas]).score(Xti, y_test)),
    ("c50py sin huecos (referencia)", C5Classifier(min_samples_leaf=10).fit(Xtr, y_train.values).score(Xte, y_test.values)),
    ("CART sin huecos (referencia)", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xtr, y_train).score(Xte, y_test)),
], columns=["modelo", "acierto en prueba"]).round(3)
print(comparacion.to_string(index=False))



Los conteos con decimales en las hojas son la propagación fraccionaria a la vista: con un 20% de
huecos la poda deja un árbol de dos hojas, porque la información que queda es poca y repartida.
En cuanto al acierto, con 73 empresas de prueba las diferencias son de dos a seis empresas y no
permiten declarar un ganador: rellenar con la mediana no es una mala opción en una tabla chica,
y descartar filas incompletas sí lo es cuando quedan pocas (46 de 146). La ventaja de la
propagación fraccionaria es que no inventa un valor: conserva la incertidumbre y la muestra en la
predicción.



## 5. Poda pesimista: podar sin datos de validación

La poda por costo-complejidad del Notebook 3 necesita datos que el árbol no vio. Quinlan poda
usando solo el entrenamiento: reconoce que el error de entrenamiento de una hoja es optimista y lo
corrige hacia arriba. Una hoja con $N$ casos y $E$ errores se trata como una muestra binomial, y se
le cobra el valor más alto del error verdadero que todavía es compatible con lo observado al nivel
de confianza `cf`. En el código de C4.5 esa cantidad se llama `AddErrs`:

$$
E_{\text{pes}} = E + \text{AddErrs}(N, E), \qquad \text{AddErrs}(N, 0) = N\,(1 - cf^{1/N})
$$

y para $E > 0$ es el límite superior del intervalo binomial con el $z$ de la tabla de C4.5 (0,69
para el defecto 0,25; 1,28 para 0,10; 2,33 para 0,01). Lo importante está en la fórmula para
$E = 0$: una hoja **pura** también paga. Una hoja de una sola observación sin errores recibe 0,75
errores pesimistas con `cf = 0,25`; una de tres, 1,11. Ese es el mecanismo que permite podar hojas
que solo memorizaron. Para cada nodo interno se suman los errores pesimistas de sus hojas y se
comparan con los del nodo convertido en hoja; si como hoja no es peor (con una tolerancia de 0,1),
se poda, de abajo hacia arriba.

Esta es la poda de la versión 0.3.0 de `c50py`, que sigue el código de C4.5. La versión anterior
usaba una aproximación normal que dejaba las hojas puras sin castigo: el mismo árbol sobre las 146
empresas quedaba con 32 hojas y 100% en entrenamiento. Ahora:


In [ ]:

filas = []
for cf in [0.01, 0.10, 0.25, 0.50]:
    for msl in [1, 5]:
        m = c5(min_samples_leaf=msl, cf=cf)
        filas.append((cf, msl, len(m.export_rules()), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
m = c5(min_samples_leaf=1, pruning=False)
filas.append(("sin poda", 1, len(m.export_rules()), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["cf", "min_samples_leaf", "hojas", "acierto entrenamiento", "acierto prueba"]).to_string(index=False))



Con estos datos el árbol sin poda ya es chico (las reglas de C4.5 para elegir cortes lo frenan
antes), así que la poda tiene poco que hacer; el `cf` más exigente (0,01) quita una hoja. La cuenta
a mano, sobre un nodo real del árbol: "deuda sobre activos ≤ 0,374", con 4 empresas (3 pagan, 1 en
impago) partidas en una hoja pura de 1 y una hoja pura de 3.


In [ ]:

from c50py.tree import _add_errs          # la función AddErrs de C4.5, tal como está en el paquete
for cf in [0.25, 0.10, 0.05, 0.01]:
    como_hoja = 1 + _add_errs(4, 1, cf)                          # el nodo como hoja: 1 error observado
    subarbol = _add_errs(1, 0, cf) + _add_errs(3, 0, cf)         # dos hojas puras: 0 errores observados
    print(f"cf = {cf:.2f}   nodo como hoja {como_hoja:.2f}   subárbol {subarbol:.2f}   ->", "se poda" if como_hoja <= subarbol + 0.1 else "se mantiene")



Con `cf = 0,25` el subárbol se mantiene (2,19 contra 1,86: las dos hojas puras pagan 0,75 y 1,11);
con `cf = 0,01` el castigo a las hojas chicas crece más rápido que el del nodo y se poda. Es
exactamente lo que muestra la tabla anterior.

**Una propiedad de esta poda que conviene tener presente.** Mira el error, no el orden. En un
problema desbalanceado como el de churn (20% de fuga), muchas hojas predicen la clase mayoritaria
y solo afinan la probabilidad; para el error de clasificación no aportan nada, y C4.5 las quita.
Por eso en la sección 1 el árbol sin poda ordenaba mejor a los clientes (AUC 0,767) que el podado
(0,716), aunque acertara menos. Si el uso es un ranking de riesgo, `pruning=False` con un mínimo
por hoja razonable; si el uso son reglas y decisiones, la poda por defecto.



## 6. Boosting incorporado

C5.0 trae el boosting adentro: `trials = 10` entrena diez árboles en serie con el esquema de
AdaBoost del Notebook 4. Cada árbol se entrena con pesos por empresa, se calcula su error
ponderado $\varepsilon$, su peso en la votación es $\alpha = \tfrac{1}{2} \ln \frac{1 - \varepsilon}{\varepsilon}$,
las empresas mal clasificadas se multiplican por $e^{\alpha}$, se renormaliza y se sigue. Se
detiene antes si un árbol es perfecto (no hay nada que corregir) o si es peor que una moneda.

Con los valores por defecto el primer árbol acierta 82,9% en entrenamiento, así que le quedan
errores que corregir y el boosting tiene trabajo desde el inicio.


In [ ]:

filas = []
for t in [1, 3, 5, 10, 25]:
    b = c5(trials=t, random_state=0)
    n_arboles = len(b.ensemble_) if hasattr(b, "ensemble_") else 1
    filas.append((t, n_arboles, round(b.score(Xtr, y_train), 3), round(b.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["trials", "árboles entrenados", "acierto entrenamiento", "acierto prueba"]).to_string(index=False))

b10 = c5(trials=10, random_state=0)
print("\nAlfas de las diez rondas:", np.round(b10.alphas_, 3))
print("Errores ponderados que implican (e = 1 / (1 + exp(2 alfa))):", np.round(1 / (1 + np.exp(2 * np.array(b10.alphas_))), 3))



Diez rondas llevan el acierto en prueba de 76,7% a 82,2%, lo mismo que el CART de profundidad 2,
y el entrenamiento a 100%: de ahí en adelante no queda nada que corregir y más rondas no cambian
nada. Como en el Notebook 4, el número de rondas es una perilla que conviene elegir con validación.

## 7. También hay un `C5Regressor`

La misma interfaz para respuestas numéricas, con la varianza como impureza. Predecimos `roa` a
partir de los otros cuatro ratios, como en el Notebook 2.


In [ ]:

SIN_ROA = [r for r in RATIOS if r != "roa"]
reg = C5Regressor(min_samples_leaf=10, numeric_threshold_strategy="all")
reg.fit(X_train[SIN_ROA].values.astype(float), X_train["roa"].values, feature_names=SIN_ROA)
reg.print_tree(feature_names=SIN_ROA) if hasattr(reg, "print_tree") else None
ref = DecisionTreeRegressor(min_samples_leaf=10, random_state=0).fit(X_train[SIN_ROA], X_train["roa"])
print(f"\nR2 en prueba: c50py {reg.score(X_test[SIN_ROA].values.astype(float), X_test['roa'].values):.3f}   CART {ref.score(X_test[SIN_ROA], X_test['roa']):.3f}")



## 8. Para cerrar

C5.0 y CART son dos respuestas a las mismas preguntas: cómo elegir el corte, qué hacer con
categorías y con huecos, cuándo parar. Ninguna es "la correcta". Sobre 219 empresas los dos dan
árboles distintos y aciertos que no se distinguen con 73 de prueba; lo que sí distingue a C5.0 es
lo que entrega alrededor del árbol: categorías sin dummies, huecos sin imputar, reglas en texto y
la regla exacta de cada predicción. Conocer las dos ramas es lo que permite leer con criterio lo
que entrega cualquier librería.

## Ejercicios

1. **Gain ratio y el identificador.** Agrega a las 14 empresas una columna `id` con un valor
   distinto por empresa y calcula, con `ganancia_y_ratio`, la ganancia y el gain ratio de partir
   por `id` en 14 grupos. Compara con deuda. ¿El gain ratio resuelve la trampa del identificador?

2. **Categorías en las 219.** Convierte `ln_activos` en tres categorías (chica, mediana, grande
   por terciles) y entrena `c50py` con esa columna como categórica y las otras cuatro numéricas.
   ¿Qué subconjunto elige en el primer corte que usa el tamaño? ¿Cambia el acierto?

3. **Cuánto falta es demasiado.** Repite la comparación de la sección 3 con 10%, 30% y 50% de
   valores borrados. ¿A partir de qué punto descartar filas completas deja de ser viable? ¿Y la
   mediana?

4. **Cuánto paga una hoja pura.** Con `_add_errs`, tabula los errores pesimistas de una hoja pura
   de $N = 1, 2, 3, 5, 10, 20$ casos para `cf` 0,25 y 0,10. ¿A partir de qué tamaño una hoja pura
   "se paga sola" (su castigo es menor que un error observado)? ¿Qué dice eso sobre qué hojas
   sobreviven a la poda?

5. **Regla por empresa.** Con `predict_rule`, obtén la regla aplicada a cada empresa de prueba y
   cuenta cuántas empresas caen en cada regla y qué fracción de ellas está en impago. ¿Qué regla es
   la más "limpia" y cuál la más dudosa?



## Soluciones


In [ ]:

# 1. El identificador
cred_id = credito.copy(); cred_id["id"] = range(1, 15)
g, s, r = ganancia_y_ratio(cred_id, [cred_id["id"] == v for v in cred_id["id"]])
print(f"id: ganancia {g:.3f} (la máxima posible, H(paga) = {H(credito['paga']):.3f}), entropía de la partición {s:.3f}, gain ratio {r:.3f}")
g2, s2, r2 = ganancia_y_ratio(credito, [credito['deuda'].isin(['baja']), ~credito['deuda'].isin(['baja'])])
print(f"deuda {{baja}} contra el resto: gain ratio {r2:.3f}")
print("El gain ratio castiga al id: de 0,985 a 0,259. En el Notebook 1 eso no bastaba, porque se comparaba con la deuda partida en tres (0,185);")
print("con el corte binario {baja} contra el resto (0,338), deuda le gana al id y la trampa queda resuelta en esta tabla. No es una garantía general:")
print("con menos filas por valor la entropía de la partición del id crece menos. Por eso C4.5 agrega otra regla: solo considerar atributos con ganancia sobre el promedio.")


In [ ]:

# 2. ln_activos en tres categorías
terciles = X_train["ln_activos"].quantile([1/3, 2/3]).values
def cat_tam(v): return "chica" if v <= terciles[0] else ("mediana" if v <= terciles[1] else "grande")
X2tr = X_train.copy(); X2te = X_test.copy()
X2tr["ln_activos"] = X2tr["ln_activos"].map(cat_tam); X2te["ln_activos"] = X2te["ln_activos"].map(cat_tam)
m2 = C5Classifier(min_samples_leaf=10, numeric_threshold_strategy="all", categorical_features=["ln_activos"])
m2.fit(X2tr.values.astype(object), y_train.values, feature_names=RATIOS)
m2.print_tree(feature_names=RATIOS)
print(f"\nAcierto en prueba: {accuracy_score(y_test, m2.predict(X2te.values.astype(object))):.3f}   (con ln_activos numérico: {c5_def.score(Xte, y_test):.3f})")
print("El tamaño en tres categorías aparece como 'ln_activos in {chica}' en el segundo nivel: C5.0 agrupó mediana y grande. Con menos información (tres")
print("categorías en vez del número) el acierto baja; el corte binario por subconjuntos es útil cuando la variable ya viene como categoría, no como reemplazo de un número.")


In [ ]:

# 3. Distintas fracciones de valores faltantes
filas = []
for frac in [0.10, 0.20, 0.30, 0.50]:
    rng = np.random.default_rng(4)
    Xm = Xtr.copy(); Xm[rng.random(Xm.shape) < frac] = np.nan
    Xtm = Xte.copy(); Xtm[rng.random(Xtm.shape) < frac] = np.nan
    mediana = np.nanmedian(Xm, axis=0); Xi, Xti = np.where(np.isnan(Xm), mediana, Xm), np.where(np.isnan(Xtm), mediana, Xtm)
    completas = ~np.isnan(Xm).any(axis=1)
    a_c5 = C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values).score(Xtm, y_test.values)
    a_med = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xi, y_train).score(Xti, y_test)
    a_comp = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xm[completas], y_train.values[completas]).score(Xti, y_test) if completas.sum() >= 20 else np.nan
    filas.append((f"{frac:.0%}", int(completas.sum()), round(a_c5, 3), round(a_med, 3), round(a_comp, 3) if not np.isnan(a_comp) else "menos de 20 filas"))
print(pd.DataFrame(filas, columns=["faltantes", "filas completas (de 146)", "c50py", "CART mediana", "CART filas completas"]).to_string(index=False))
print("\nDescartar filas deja de ser viable cuando quedan pocas decenas (30% ya deja 26 filas). La mediana aguanta más, pero con 50% de huecos")
print("la mitad de cada columna es un número inventado y la propagación fraccionaria, que conserva la incertidumbre, se vuelve la opción más honesta.")


In [ ]:

# 4. Cuánto paga una hoja pura según su tamaño
tabla4 = pd.DataFrame({cf: [round(_add_errs(N, 0, cf), 2) for N in [1, 2, 3, 5, 10, 20]] for cf in [0.25, 0.10]}, index=[1, 2, 3, 5, 10, 20])
tabla4.index.name = "N de la hoja pura"; tabla4.columns = ["cf = 0,25", "cf = 0,10"]
print(tabla4)
print("\nCon cf = 0,25 una hoja pura de 3 casos ya paga 1,11 errores, más que un error observado; con 5, 1,21; con 20, 1,34: el castigo crece despacio")
print("(tiende a -ln(cf) = 1,39). Así que una hoja pura chica solo sobrevive si, al fundirla con su hermana, el nodo resultante cometería más de un error")
print("extra: la poda elimina las hojas que separan una o dos observaciones y conserva las que separan grupos.")


In [ ]:

# 5. Reglas por empresa de prueba
reglas = pd.Series(c5_def.predict_rule(Xte), index=X_test.index)
resumen = pd.DataFrame({"regla": reglas, "impago": y_test}).groupby("regla")["impago"].agg(["count", "mean"]).rename(columns={"count": "empresas de prueba", "mean": "fracción en impago"}).round(2)
print(resumen.sort_values("empresas de prueba", ascending=False).to_string())
print("\nEn prueba, las reglas más limpias son 'roa > 0,088 (con ventas/deuda > 1,278 y deuda/activos > 0,586) => impago' (4 de 4) y la regla grande")
print("'ventas/deuda > 1,278 y deuda/activos <= 0,586 => paga' (42 empresas, 14% en impago). La más dudosa es 'roa <= 0,088 => paga': de sus 11 empresas")
print("de prueba, 7 cayeron en impago; en entrenamiento era 20 contra 12 y la mayoría decidía 'paga'. Es la hoja que un comité debería mirar primero.")
